> **归档说明**：本 notebook 记录项目开发过程中的中间实验，保留用于复盘和审计，不作为最终展示入口。部分输入可能依赖本地生成但未上传 GitHub 的过程产物，例如 `outputs/predictions/`、`threshold_metrics` 或 `trial_results`。复现这些历史实验前，请先查看 `reports/notebook_reproducibility_audit.md` 中对应的再生成脚本说明。项目最终展示入口见 `notebooks/final/` 和 `README.md`。
>
> 该实验结果仅作为历史对照，不作为最终模型选择依据。


# Day 7 风险分层与业务交付总结

Day 7 的目标是把 Day 6 的阈值成本分析结果转化为业务可读的维修优先级清单、风险等级汇总和项目交付材料。本阶段不重新训练模型、不做 GridSearch、不做新的特征工程，也不把测试集回溯阈值写成生产环境最终阈值。

## 1. 读取 cfg 和依赖

继续从 `config/config.yaml` 读取路径和成本配置。

In [1]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

from scania_aps.config import get_config
from scania_aps.evaluation.metrics import evaluate_binary_classifier
from scania_aps.evaluation.risk_utils import (
    ACTION_MAPPING,
    RISK_LEVEL_ORDER,
    build_maintenance_priority_table,
)

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 140)

cfg = get_config(PROJECT_ROOT / "config" / "config.yaml")
cfg.false_positive_cost, cfg.false_negative_cost, cfg.tables_dir

(10, 500, WindowsPath('C:/Scania APS/outputs/tables'))

## 2. 读取 Day 6 最优阈值结果

当前回溯分析推荐组合为 XGBoost + `median_all`。注意：这个阈值来自测试集敏感性分析，不是生产环境最终阈值。

In [2]:
MODEL_NAME = "xgboost_scale_pos_weight"
STRATEGY = "median_all"

best_summary = pd.read_csv(cfg.metrics_dir / "day6_best_threshold_summary.csv")
best_row = best_summary[
    best_summary["model_name"].eq(MODEL_NAME)
    & best_summary["strategy"].eq(STRATEGY)
].iloc[0]

best_threshold = float(best_row["best_threshold"])
best_row.to_frame().T

,model_name,strategy,best_threshold,precision,recall,f1,f2,fp,fn,total_cost,average_precision
0,xgboost_scale_pos_weight,median_all,0.2,0.469231,0.976,0.633766,0.802632,414,9,8640,0.911301


## 3. 读取 Day 5 predictions 并选择候选方案

这里只读取已有预测概率，不重新训练 XGBoost。

In [3]:
predictions = pd.read_csv(cfg.predictions_dir / "day5_model_compare_predictions.csv")
selected_predictions = predictions[
    predictions["model_name"].eq(MODEL_NAME)
    & predictions["strategy"].eq(STRATEGY)
].copy()

selected_predictions.shape, selected_predictions.head()

((16000, 8),
       dataset  sample_id  y_true   y_proba  y_pred                model_name    strategy  threshold
 16000    test          1       0  0.002150       0  xgboost_scale_pos_weight  median_all        0.5
 16001    test          2       0  0.000286       0  xgboost_scale_pos_weight  median_all        0.5
 16002    test          3       0  0.012219       0  xgboost_scale_pos_weight  median_all        0.5
 16003    test          4       0  0.000220       0  xgboost_scale_pos_weight  median_all        0.5
 16004    test          5       0  0.002308       0  xgboost_scale_pos_weight  median_all        0.5)

## 4. 生成风险分层和维修优先级表

风险分层规则：`Critical` 为预测概率大于等于 0.80；`High` 为低成本阈值到 0.80；`Medium` 为 0.05 到低成本阈值；`Low` 为小于 0.05。

In [4]:
priority_table = build_maintenance_priority_table(
    predictions_df=selected_predictions,
    best_threshold=best_threshold,
    model_name=MODEL_NAME,
    strategy=STRATEGY,
)

priority_table.head(20)

,sample_id,model_name,strategy,y_true,y_proba,threshold,y_pred,prediction_type,risk_level,suggested_action
11828,11829,xgboost_scale_pos_weight,median_all,1,0.999773,0.2,1,TP,Critical,立即检修
9396,9397,xgboost_scale_pos_weight,median_all,1,0.999761,0.2,1,TP,Critical,立即检修
14425,14426,xgboost_scale_pos_weight,median_all,1,0.999622,0.2,1,TP,Critical,立即检修
15298,15299,xgboost_scale_pos_weight,median_all,1,0.999606,0.2,1,TP,Critical,立即检修
9795,9796,xgboost_scale_pos_weight,median_all,1,0.999535,0.2,1,TP,Critical,立即检修
13604,13605,xgboost_scale_pos_weight,median_all,1,0.999533,0.2,1,TP,Critical,立即检修
3336,3337,xgboost_scale_pos_weight,median_all,1,0.999525,0.2,1,TP,Critical,立即检修
3581,3582,xgboost_scale_pos_weight,median_all,1,0.999518,0.2,1,TP,Critical,立即检修
4405,4406,xgboost_scale_pos_weight,median_all,1,0.999484,0.2,1,TP,Critical,立即检修
11538,11539,xgboost_scale_pos_weight,median_all,1,0.999466,0.2,1,TP,Critical,立即检修


## 5. 风险等级汇总

风险等级汇总帮助维修团队理解不同优先级下的车辆数量、实际故障数量和预测表现。

In [5]:
risk_rows = []
for risk_level in RISK_LEVEL_ORDER:
    group = priority_table[priority_table["risk_level"].eq(risk_level)]
    type_counts = group["prediction_type"].value_counts()
    risk_rows.append(
        {
            "risk_level": risk_level,
            "sample_count": len(group),
            "actual_pos_count": int(group["y_true"].sum()),
            "actual_pos_rate": group["y_true"].mean() if len(group) else 0.0,
            "predicted_pos_count": int(group["y_pred"].sum()),
            "tp": int(type_counts.get("TP", 0)),
            "fp": int(type_counts.get("FP", 0)),
            "fn": int(type_counts.get("FN", 0)),
            "tn": int(type_counts.get("TN", 0)),
            "avg_y_proba": group["y_proba"].mean() if len(group) else 0.0,
            "suggested_action": ACTION_MAPPING[risk_level],
        }
    )

risk_summary = pd.DataFrame(risk_rows)
risk_summary

,risk_level,sample_count,actual_pos_count,actual_pos_rate,predicted_pos_count,tp,fp,fn,tn,avg_y_proba,suggested_action
0,Critical,411,324,0.788321,411,324,87,0,0,0.960886,立即检修
1,High,369,42,0.113821,369,42,327,0,0,0.442347,优先检修
2,Medium,415,6,0.014458,0,0,0,6,409,0.100695,观察复查
3,Low,14805,3,0.000203,0,0,0,3,14802,0.003363,暂不处理


## 6. 成本结果汇总

naive baseline 指全部预测为 `neg`。在 FN 成本远高于 FP 的场景下，当前方案允许一定 FP 增加，以换取 FN 大幅下降和总成本降低。

In [6]:
metric = evaluate_binary_classifier(
    y_true=priority_table["y_true"],
    y_pred=priority_table["y_pred"],
    y_proba=priority_table["y_proba"],
    cfg=cfg,
    model_name=MODEL_NAME,
    strategy=STRATEGY,
    threshold=best_threshold,
)

naive_total_cost = int(priority_table["y_true"].sum()) * cfg.false_negative_cost
business_summary = pd.DataFrame(
    [
        {
            "solution_name": f"{MODEL_NAME}_{STRATEGY}_threshold_{best_threshold:.2f}",
            "threshold": best_threshold,
            "precision": metric["precision"],
            "recall": metric["recall"],
            "f2": metric["f2"],
            "fp": metric["fp"],
            "fn": metric["fn"],
            "total_cost": metric["total_cost"],
            "baseline_total_cost": naive_total_cost,
            "cost_reduction": naive_total_cost - metric["total_cost"],
            "cost_reduction_rate": (naive_total_cost - metric["total_cost"]) / naive_total_cost,
        }
    ]
)

business_summary

,solution_name,threshold,precision,recall,f2,fp,fn,total_cost,baseline_total_cost,cost_reduction,cost_reduction_rate
0,xgboost_scale_pos_weight_median_all_threshold_...,0.2,0.469231,0.976,0.802632,414,9,8640,187500,178860,0.95392


## 7. 默认阈值、低成本阈值和 naive baseline 对比

默认 0.5 阈值并不是成本最低选择。阈值 0.20 下 FP 增加，但 FN 从 33 降到 9，因此 total cost 明显下降。

In [7]:
day5_metrics = pd.read_csv(cfg.metrics_dir / "day5_model_compare_metrics.csv")
default_xgb = day5_metrics[
    day5_metrics["model_name"].eq(MODEL_NAME)
    & day5_metrics["strategy"].eq(STRATEGY)
].iloc[0]

comparison = pd.DataFrame(
    [
        {
            "方案": "Naive baseline，全预测 neg",
            "阈值": None,
            "Precision": 0.0,
            "Recall": 0.0,
            "F2": 0.0,
            "FP": 0,
            "FN": int(priority_table["y_true"].sum()),
            "Total Cost": naive_total_cost,
        },
        {
            "方案": "XGBoost + median_all 默认阈值",
            "阈值": default_xgb["threshold"],
            "Precision": default_xgb["precision"],
            "Recall": default_xgb["recall"],
            "F2": default_xgb["f2"],
            "FP": int(default_xgb["fp"]),
            "FN": int(default_xgb["fn"]),
            "Total Cost": default_xgb["total_cost"],
        },
        {
            "方案": "XGBoost + median_all 低成本阈值",
            "阈值": best_threshold,
            "Precision": metric["precision"],
            "Recall": metric["recall"],
            "F2": metric["f2"],
            "FP": metric["fp"],
            "FN": metric["fn"],
            "Total Cost": metric["total_cost"],
        },
    ]
)

comparison

,方案,阈值,Precision,Recall,F2,FP,FN,Total Cost
0,Naive baseline，全预测 neg,NaN,0.000000,0.000,0.000000,0,375,187500
1,XGBoost + median_all 默认阈值,0.5,0.635688,0.912,0.839058,196,33,18460
2,XGBoost + median_all 低成本阈值,0.2,0.469231,0.976,0.802632,414,9,8640


## 8. 保存 Day 7 输出

这些 CSV 可以后续导入 MySQL，用 SQL 复核风险分层和维修工作量。

In [8]:
cfg.tables_dir.mkdir(parents=True, exist_ok=True)

priority_path = cfg.tables_dir / "day7_maintenance_priority_list.csv"
risk_summary_path = cfg.tables_dir / "day7_risk_level_summary.csv"
business_summary_path = cfg.tables_dir / "day7_business_result_summary.csv"

priority_table.to_csv(priority_path, index=False, encoding="utf-8-sig")
risk_summary.drop(columns=["actual_pos_rate"]).to_csv(risk_summary_path, index=False, encoding="utf-8-sig")
business_summary.to_csv(business_summary_path, index=False, encoding="utf-8-sig")

priority_path, risk_summary_path, business_summary_path

(WindowsPath('C:/Scania APS/outputs/tables/day7_maintenance_priority_list.csv'),
 WindowsPath('C:/Scania APS/outputs/tables/day7_risk_level_summary.csv'),
 WindowsPath('C:/Scania APS/outputs/tables/day7_business_result_summary.csv'))

## 9. Day 7 小结与项目局限

- 当前候选方案为 XGBoost + `median_all` + 阈值 0.20。
- 该组合在测试集回溯分析中将 total cost 从 naive baseline 的 187,500 降到 8,640。
- 风险分层可以把车辆转化为 `立即检修`、`优先检修`、`观察复查` 和 `暂不处理` 四类维修动作。
- 当前阈值来自测试集回溯分析，不是生产环境最终阈值。
- 数据集较老、特征匿名、缺少时间戳和真实车辆 ID，因此还不能直接作为线上系统结论。
- 如果真实上线，应使用时间切分验证集选择阈值，并结合维修团队容量、误报承受能力和生产监控持续校准。